<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## Classification problem on molecular graphs using Graph Neural Networks (GNN) and `pytorch-geometric`

**Goal** Familiarize `pytorch-geometric` in handling GNNs and `DataLoaders`, and classify whether a molecule is toxic or not (**molecular level binary-property**)

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training dataset** : goal is to predict chemical toxicity using Graph Neural Networks

* ~7,800 molecules represented by SMILES strings, each with the outputs from 12 binary classification assays. Labels are either 1 = active, 0 = inactive or NaN = not tested

* Assays include nuclear receptor signaling pathways (7 assays) and stress response pathways (5 assays)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Tools**
* `scikit-learn`, `torch`
* **Core cheminfo**: `RDKit` **fingerprints, descriptors** to automate molecular representation and generate the input to the classifier
* `pytorch-geometric` to define the GNN layers, the head of network is a classifier
* `torch` to train and test, using **batches** and **early_stopping**
</div>

In [1]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.1+cu118.html
!pip install torch-geometric

!pip install rdkit-pypi

Looking in links: https://data.pyg.org/whl/torch-2.0.1+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 54.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.5/886.5 kB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 459.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 52.4 MB/s eta 0:00:00


In [23]:
import pandas as pd
import numpy as np
import torch, os, joblib, sys
import torch.nn as nn
import torch.nn.functional as F    # activation functions
from torch.utils.data import random_split
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv
import copy

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, AllChem

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib
%matplotlib inline

import matplotlib.pyplot as plt

from utils import mol_to_nx, visualize_molecular_graph, atom_features, bond_features, graph_featurizer_pygeom
from utils import split_data

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib
%matplotlib inline

import matplotlib.pyplot as plt

from utils import mol_to_nx, visualize_molecular_graph, atom_features, bond_features, graph_featurizer_pygeom
from utils import split_data

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Read in Tox21 dataset, pick one classification task and featurize (calculate molecular graph embeddings as `geometric` Data objects)
</div>

In [14]:
# Tox21 da MoleculeNet, downloaded from the internet, sicne the original link/url comes with restrictions
df = pd.read_csv('tox21.csv')

# Print columns
print(list(df.columns))  # print first few databse columns (SMILES + target)

print('Number of molecules in dataset = ' + str(df.shape[0]))
print('Number of assays available for classification tasks = ' + str(df.shape[1]-2))

['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']
Number of molecules in dataset = 7831
Number of assays available for classification tasks = 12


In [15]:
task = 'NR-AR'
if task not in list(df.columns):
    print('acthung! there is no such classification task in the input file')

In [18]:
def graph_featurizer_pygeom(mol, mol_id, y, edge_attrib = None):

    """A molecule is translated into a featurized graphs, with nodes and node labels + edges and edge labels (if applicable) """

    atom_feats = []
    edge_index = []
    edge_attr = []

    for atom in mol.GetAtoms():
        atom_feats.append(atom_features(atom))     # list of torch tensors

    for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            edge_index.append([i, j])
            edge_index.append([j, i])  # graph is undirected

            if edge_attrib is not None:
                edge_attr.append(bond_features(bond))
                edge_attr.append(bond_features(bond)) # if bidirectional, we miss half the bonds if we don't include the flipped one
                                                  # clearly here [i,j] and [j,i] share the same feature
    # from list of torch tensors to one torch tensor
    x = torch.stack(atom_feats)

    # from a list of numpy arrays to a torch tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).T   # now we need to transpose this for compatibility sake

    #print(x.shape)    # (number of atoms, number of features)
    #print(edge_index.shape)    # (2, number of edges * 2), each edge is listed twice (i,j and j,i)
    #print(edge_attr.shape)    # (number of edges * 2, size of one-hot encoding for edge embeddings, if applicable

    if edge_attrib is not None:
        edge_attr = torch.stack(edge_attr)
        data = Data(x=x, edge_index=edge_index, edge_attr = edge_attr, y=y)     # this is a torch-geometric dataset format, compatible with NNs syntax
    else:
        data = Data(x=x, edge_index=edge_index, y=y)
    data.idx = torch.tensor([mol_id])     # keep track of mol_id, since later shuffling

    return data

In [19]:
from torch_geometric.data import Data, DataLoader

full_data = []
for k, smile in enumerate(df['smiles'].values):
  # check if that task label is present or not (some assays were inconclusive on some molecules)
  if np.isnan(df[task].values[k]) == False:
    full_data.append(graph_featurizer_pygeom(Chem.MolFromSmiles(smile), k, df[task].values[k], edge_attrib=None))

[23:50:10] WARNING: not removing hydrogen atom without neighbors
/tmp/ipython-input-3254083577.py:26: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3725.)
  edge_index = torch.tensor(edge_index, dtype=torch.long).T   # now we need to transpose this for compatibility sake


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Define train, test and validation datasets, all organized in batches

</div>

In [22]:
# split into train, test, validation set using pytorch functionalities
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
n_batches = 25

train_loader, val_loader, test_loader = split_data(full_data, train_ratio, val_ratio, n_batches)

NameError: name 'random_split' is not defined

In [ ]:
in_dim = train_dataset[0][0].shape[1]
print('Embedding size of nodes = ' + str(in_dim))
print('Number of samples to ne used in training = ' + str(len(train_dataset))

Data(x=[32, 5], edge_index=[2, 70], y=1.0, idx=[1])


In [ ]:
class GNNclassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        self.conv1 = GraphConv(in_dim, hidden_dim)     # this is aready a GNN layer that performs message passing with the nearest neighobrs
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.classifier = torch.nn.Linear(hidden_dim, 1)  # single Linear, we can always make this more complex

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)    # the linear layer is already implemented
        x = F.relu(x) # just need to apply a non linear activation function
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # graph embedding, averaging over all the noves
        x = self.classifier(x)     # logits per graph, shape [batch_size, 1]
        return x

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Instantiate model, optimizer and Binary Cross Entropy (w Logits) function; then train the model using early stopping
Please note the model we chose (GNVConv) cannot handle edge message passing
</div>

In [ ]:
model = GNNclassifier(in_dim=n=in_dim, hidden_dim=8) # in_dim is hard_coded, we might automate this based on the features
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCEWithLogitsLoss()

# Early stopping setup
patience = 10
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0
n_epochs = 200

for epoch in range(n_epochs):

    model.train()
    total_train_loss = 0

    # train model
    for batch in train_loader:
        optimizer.zero_grad()

        # extract features ('X') and labels ('y')
        pred = model(batch.x, batch.edge_index, batch.batch)
        labels = batch.y.float().unsqueeze(1)

        # compute the loss function and backpropagate using the user-specified optimizer
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()

        # add loss from this batch to the total training loss
        total_train_loss += loss.item()

    if epoch % 10 == 0 or epoch == 49:

        # model optimized for now, ready to propagate forward
        model.eval()
        with torch.no_grad():

            total_correct = 0
            total = 0
            total_val_loss = 0

            for val_batch in val_loader:

                # extract the input and output for the batch, compute the loss for the batch, add that to the full validation loss
                pred = torch.sigmoid(model(val_batch.x, val_batch.edge_index, val_batch.batch)).squeeze(1)
                labels = val_batch.y.view(-1)

                val_loss = loss_fn(pred, labels)
                total_val_loss += val_loss.item()

                # classify: map probabilities to binary 0 and 1
                predicted = (pred > 0.5).float().view(-1)
                total += labels.size(0)

                total_correct += (predicted == labels).sum().item() # count those predictions that match the known labels

            # average loss on the validation dataset
            avg_val_loss = total_val_loss / len(val_loader)
            val_acc = total_correct / total

            print(f"Epoch {epoch} | Train Loss: {total_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}")

        # === Early stopping check ===
        if avg_val_loss < best_val_loss:     # if performance on validation set is still improving, keep going
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())    # save model parameters
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:    # performance on validation set stopped improving
                print(f"Early stopping at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
                break

# Load best weights
model.load_state_dict(best_model_state)

GNNclassifier(
  (conv1): GraphConv(5, 8)
  (conv2): GraphConv(8, 8)
  (classifier): Linear(in_features=8, out_features=1, bias=True)
)
Epoch 0 | Train Loss: 47.2559 | Val Loss: 0.7139 | Val Acc: 0.96
Epoch 10 | Train Loss: 27.8026 | Val Loss: 0.7164 | Val Acc: 0.97
Epoch 20 | Train Loss: 25.7567 | Val Loss: 0.7004 | Val Acc: 0.97
Epoch 30 | Train Loss: 24.6291 | Val Loss: 0.6999 | Val Acc: 0.97
Epoch 40 | Train Loss: 24.8103 | Val Loss: 0.7123 | Val Acc: 0.97
Epoch 49 | Train Loss: 24.5066 | Val Loss: 0.7099 | Val Acc: 0.97
Epoch 50 | Train Loss: 24.7646 | Val Loss: 0.7038 | Val Acc: 0.97
Epoch 60 | Train Loss: 24.6216 | Val Loss: 0.7010 | Val Acc: 0.97
Epoch 70 | Train Loss: 24.2355 | Val Loss: 0.7084 | Val Acc: 0.97
Epoch 80 | Train Loss: 24.0994 | Val Loss: 0.7026 | Val Acc: 0.97
Epoch 90 | Train Loss: 23.8695 | Val Loss: 0.7104 | Val Acc: 0.97
Epoch 100 | Train Loss: 24.0534 | Val Loss: 0.7015 | Val Acc: 0.97
Epoch 110 | Train Loss: 23.7123 | Val Loss: 0.7053 | Val Acc: 0.97
Epoch

<All keys matched successfully>

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Test trained GNN optimizer on the test set molecular graphs, compute metrics
</div>

In [ ]:
# Set model in evaluation mode
model.eval()

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
  for batch in test_loader:
      pred = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
      predicted = (pred > 0.5).float().view(-1)
      all_preds.append(predicted)
      all_labels.append(batch.y)

# now we are going to use some scikit-learn functions that only take in NumPy arrays. We need to convert tensors then.
y_pred = torch.cat(all_preds, dim=0).cpu().numpy()
y_true = torch.cat(all_labels, dim = 0).cpu().numpy()

# Compute assessment metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1 Score:  {f1:.3f}")
print(f"AUC:       {auc:.3f}")

# we can then build the roc-auc plot

Accuracy:  0.967
Precision: 0.704
Recall:    0.404
F1 Score:  0.514
AUC:       0.698


In [ ]:
plt.plot(y_true, y_pred, 'o')

NameError: name 'plt' is not defined